## TypedDict

第二推荐的结构化输出方式

TypedDict 是 Python 3.8+ 引入的一种类型提示工具，即带有类型声明的字典结构。适合需要快速定义字典结构且无需 Pydantic 重量级功能的场景。

相比于Dict，TypedDict 可以进一步说明：
- 这个字典应该有哪些字段
- 每个字段的类型是什么
- TypedDict 主要是类型声明，不是运行时强校验器。

IDE的静态类型检查会标记出类型错误。

Annotated 用来在“类型”之外，再附加一些额外信息，即元数据。类似于Pydantic的Field。

### 基本用法

In [6]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import os
from typing_extensions import TypedDict, Annotated

llm = init_chat_model(
    model="deepseek-v4-flash",
    extra_body={"thinking": {"type": "disabled"}} # Deepseek 在思考模式下不支持结构化输出
)


class MovieTypedDict(TypedDict):
    """
    电影的详细信息
    """
    title: Annotated[str, "电影的正式名称，例如《盗梦空间》"]
    year: Annotated[int, "电影的公映年份，使用四位数字表示"]
    director: Annotated[str, "电影导演的全名"]
    rating: Annotated[float, "电影在10分制下的评分，可包含一位小数"]

# 设置模型结构化输出
structured_llm  = llm.with_structured_output(MovieTypedDict)
# 调用模型并获取结构化输出
response = structured_llm.invoke("给我介绍下电影《星际穿越》")
print(type(response))
print(response)


<class 'dict'>
{'title': '星际穿越', 'year': 2014, 'director': '克里斯托弗·诺兰', 'rating': 9.4}


In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from pydantic import BaseModel, Field
import os
from typing import List
from typing_extensions import TypedDict, Annotated

llm_ds = init_chat_model(
    model="deepseek-v4-pro",
    extra_body={"thinking": {"type": "disabled"}} # Deepseek 在思考模式下不支持结构化输出
)

llm_gpt = init_chat_model(
    model="gpt-4o",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_BASE_URL"),
)
# 使用TypedDict定义嵌套结构
class Actor(TypedDict):
    """演员情况"""
    name: Annotated[str, "演员姓名"]
    role: Annotated[str, "饰演的角色"]
class Movie(TypedDict):
    """电影情况"""
    title: Annotated[str, "电影标题"]
    year: Annotated[int, "上映年份"]
    director: Annotated[str, "导演"]
    cast: Annotated[List[Actor], "演员列表"]  # 嵌套列表定义
    rating: Annotated[float, "评分"]
# 设置模型结构化输出
structured_llm_ds  = llm_ds.with_structured_output(Movie)
# 调用模型并获取结构化输出
resp_ds = structured_llm_ds.invoke("给我介绍下电影《盗梦空间》")


structured_llm_gpt  = llm_gpt.with_structured_output(Movie)

resp_gpt = structured_llm_gpt.invoke("给我介绍下电影《盗梦空间》")

print(resp_gpt) # TypedDict GPT可以结构化输出

print(resp_ds) # TypedDict deepseek-v4-pro 和 deepseek-v4-flash 都不可以结构化输出



{'title': '盗梦空间', 'year': 2010, 'director': '克里斯托弗·诺兰', 'cast': [{'name': '莱昂纳多·迪卡普里奥', 'role': '多姆·柯布'}, {'name': '艾伦·佩吉', 'role': '阿里阿德妮'}, {'name': '约瑟夫·高登-莱维特', 'role': '亚瑟'}, {'name': '汤姆·哈迪', 'role': '伊姆斯'}, {'name': '玛丽昂·歌迪亚', 'role': '玛尔'}, {'name': '渡边谦', 'role': '斋藤'}], 'rating': 8.8}
{'title': '盗梦空间'}


### ...的使用

...是Python的字面量，等价于Ellipsis，可以理解为占位符。下游框架（如LangChain）可以对...作定制化处理，如LangChain中Annotated的...表示当前字段是必须存在的，
不可省略，用来指示模型的输出。

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rich_print
from typing import List
from typing_extensions import TypedDict, Annotated
import os

load_dotenv(override=True)


class ActorDict(TypedDict):
    """演员信息"""
    name: Annotated[str, ..., "演员姓名"]
    role: Annotated[str, ..., "饰演的角色"]


class MovieDict(TypedDict):
    """
    电影的详细信息

    这里继续保留 Annotated[..., "说明"] 的写法：
    ... / Ellipsis 表示该字段是结构化输出里的必填字段。
    """
    title: Annotated[str, ..., "电影标题"]
    year: Annotated[int, ..., "电影上映年份"]
    director: Annotated[str, ..., "导演"]
    cast: Annotated[List[ActorDict], ..., "演员列表"]
    rating: Annotated[float, ..., "电影评分，满分十分"]


def validate_movie_result(result: dict) -> list[str]:
    """TypedDict 只提供结构声明，本身不会在运行时生成实例校验器，所以这里手动校验返回 dict。"""
    errors = []

    if not isinstance(result, dict):
        return [f"返回值不是 dict，而是 {type(result).__name__}"]

    required_fields = ["title", "year", "director", "cast", "rating"]
    for field in required_fields:
        if field not in result:
            errors.append(f"缺少必填字段: {field}")

    if "title" in result and not isinstance(result["title"], str):
        errors.append("title 应该是 string")
    if "year" in result and (not isinstance(result["year"], int) or isinstance(result["year"], bool)):
        errors.append("year 应该是 integer")
    if "director" in result and not isinstance(result["director"], str):
        errors.append("director 应该是 string")
    if "rating" in result and (not isinstance(result["rating"], (int, float)) or isinstance(result["rating"], bool)):
        errors.append("rating 应该是 number")

    cast = result.get("cast")
    if "cast" in result and not isinstance(cast, list):
        errors.append("cast 应该是 array")
    elif isinstance(cast, list):
        for index, actor in enumerate(cast):
            if not isinstance(actor, dict):
                errors.append(f"cast[{index}] 应该是 object")
                continue
            if not isinstance(actor.get("name"), str):
                errors.append(f"cast[{index}].name 应该是 string")
            if not isinstance(actor.get("role"), str):
                errors.append(f"cast[{index}].role 应该是 string")

    return errors


models = {
    "deepseek-v4-flash": init_chat_model(
        model="deepseek-v4-flash",
        extra_body={"thinking": {"type": "disabled"}},
    ),
    "deepseek-v4-pro": init_chat_model(
        model="deepseek-v4-pro",
        extra_body={"thinking": {"type": "disabled"}},
    ),
    "gpt-5.4-mini": init_chat_model(
        model="gpt-5.4-mini",
        model_provider="openai",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        base_url=os.getenv("OPENROUTER_BASE_URL"),
    ),
}


# 使用较弱提示词，不主动列出字段，用来测试模型/适配器是否真的会按 schema 补齐必填字段。
prompt = "请介绍电影《盗梦空间》"

for model_name, llm in models.items():
    print(f"\n===== {model_name} =====")
    try:
        structured_model = llm.with_structured_output(MovieDict)
        response = structured_model.invoke(prompt)
        errors = validate_movie_result(response)

        if errors:
            print("[FAIL] TypedDict 结构校验失败")
            rich_print(errors)
        else:
            print("[PASS] TypedDict 结构校验通过")

        rich_print(response)

    except Exception as e:
        print("[FAIL] 调用失败")
        print(type(e).__name__, e)


## JSON Schema

该方式手动按照JSON Schema规范拼接JSON字符串，编写繁琐，缺少原生校验机制，**不推荐使用**。


1. **`method="json_schema"`**
指定结构化输出实现方案，可用性取决于大模型厂商与LangChain适配器适配情况，例如DeepSeek部分服务不支持该模式；该参数代表调用模型厂商原生专用结构化输出接口。

2. JSON Schema 标准固定关键字释义

| 关键字 | 作用 |
|--------|------|
| `title` | 给Schema/属性设置可读标题，**不建议中文**，用于可读性 |
| `description` | 对整体结构或单个字段做详细说明，辅助大模型理解字段含义 |
| `type` | 约束数据类型：`string`/`number`/`integer`/`boolean`/`object`/`array`/`null` |
| `properties` | 仅用于`object`类型，定义对象内所有键名、对应值类型与描述 |
| `required` | 仅用于`object`类型，数组形式列出**必填字段**，缺失则不符合规范 |


In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rich_print
import os

load_dotenv(override=True)


# 1. 定义嵌套的 JSON Schema
project_schema = {
    "title": "MovieInfo",
    "description": "包含电影标题、上映年份、导演、演员和评分的电影对象",
    "type": "object",
    "properties": {
        "title": {"type": "string", "description": "电影标题"},
        "year": {"type": "integer", "description": "上映年份"},
        "director": {"type": "string", "description": "导演"},
        "cast": {  # 定义嵌套数组
            "type": "array",
            "description": "演员列表",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "演员姓名"},
                    "role": {"type": "string", "description": "演员角色"}
                },
                "required": ["name", "role"]
            }
        },
        "rating": {"type": "number", "description": "评分（10分制）"}
    },
    "required": ["title", "year", "director", "cast", "rating"]
}

def validate_movie_result(result: dict) -> list[str]:
    """对模型返回结果做一层 Python 侧校验，确认它确实符合上面的 JSON Schema 形状。"""
    errors = []

    if not isinstance(result, dict):
        return [f"返回值不是 dict，而是 {type(result).__name__}"]

    required_fields = ["title", "year", "director", "cast", "rating"]
    for field in required_fields:
        if field not in result:
            errors.append(f"缺少必填字段: {field}")

    if "title" in result and not isinstance(result["title"], str):
        errors.append("title 应该是 string")
    if "year" in result and (not isinstance(result["year"], int) or isinstance(result["year"], bool)):
        errors.append("year 应该是 integer")
    if "director" in result and not isinstance(result["director"], str):
        errors.append("director 应该是 string")
    if "rating" in result and (not isinstance(result["rating"], (int, float)) or isinstance(result["rating"], bool)):
        errors.append("rating 应该是 number")

    cast = result.get("cast")
    if "cast" in result and not isinstance(cast, list):
        errors.append("cast 应该是 array")
    elif isinstance(cast, list):
        for index, actor in enumerate(cast):
            if not isinstance(actor, dict):
                errors.append(f"cast[{index}] 应该是 object")
                continue
            if not isinstance(actor.get("name"), str):
                errors.append(f"cast[{index}].name 应该是 string")
            if not isinstance(actor.get("role"), str):
                errors.append(f"cast[{index}].role 应该是 string")

    return errors


models = {
    "deepseek-v4-flash": init_chat_model(
        model="deepseek-v4-flash",
        extra_body={"thinking": {"type": "disabled"}},  # DeepSeek 思考模式下不稳定，验证结构化输出时关闭。
    ),
    "deepseek-v4-pro": init_chat_model(
        model="deepseek-v4-pro",
        extra_body={"thinking": {"type": "disabled"}},
    ),
    "gpt-5.4-mini": init_chat_model(
        model="gpt-5.4-mini",
        model_provider="openai",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        base_url=os.getenv("OPENROUTER_BASE_URL"),
    ),
}


# 使用较弱提示词，不主动列出字段，用来测试模型/适配器是否真的会按 schema 补齐必填字段。
prompt = "请介绍电影《盗梦空间》"

for model_name, llm in models.items():
    print(f"\n===== {model_name} =====")
    try:
        # method='json_schema' 会让 LangChain 尽量使用模型供应商的 JSON Schema / 结构化输出能力。
        structured_model = llm.with_structured_output(schema=project_schema, method="json_schema")
        response = structured_model.invoke(prompt)
        errors = validate_movie_result(response)

        if errors:
            print("[FAIL] JSON Schema 结构校验失败")
            rich_print(errors)
        else:
            print("[PASS] JSON Schema 结构校验通过")

        rich_print(response)

    except Exception as e:
        print("[FAIL] 调用失败")
        print(type(e).__name__, e)


## @dataclass

`@dataclass` 是 Python 标准库 `dataclasses` 提供的类装饰器，用来简化**以字段为核心**的数据类定义。

给类添加 `@dataclass` 装饰器后，Python 会根据你声明的类字段，**自动生成以下常用内置方法**：
- `__init__`：构造初始化方法
- `__repr__`：打印对象时的格式化输出方法
- `__eq__`：对象相等性比较方法

从使用表现来看，`@dataclass` 生成的类，效果**近似于**手动手写上述方法的普通Python类。

```python
from dataclasses import dataclass

@dataclass
class Movie:
    title: str
    year: int
    director: str
    rating: float
```

该写法核心优势：**数据结构定义代码更简洁、可读性更强**。

1. 虽然 `@dataclass` 修饰的类行为和手动写 `__init__`/`__repr__`/`__eq__` 的普通类很像，但**二者并不完全等价**。
2. `@dataclass` 修饰后依旧是标准Python类，但会被标准库标记为**数据类**，并且会附带保存字段元信息；纯手动编写初始化等魔法方法的普通类，无法替代它。
3. 重要区别：**被 `@dataclass` 修饰的数据类可以直接作为 LangChain 的 Schema 用于结构化输出，纯手写初始化方法的普通类不能**。

In [ ]:
from dataclasses import asdict, dataclass, field, is_dataclass
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
from rich import print as rich_print
from typing import List
import os

load_dotenv(override=True)


@dataclass
class Actor:
    """演员信息"""
    name: str = field(metadata={"description": "演员姓名"})
    role: str = field(metadata={"description": "饰演的角色"})


@dataclass
class Movie:
    """电影的详细信息"""
    title: str = field(metadata={"description": "电影标题"})
    year: int = field(metadata={"description": "电影上映年份"})
    director: str = field(metadata={"description": "导演"})
    cast: List[Actor] = field(metadata={"description": "演员列表"})
    rating: float = field(metadata={"description": "电影评分，满分十分"})


def normalize_movie_result(result):
    """dataclass 成功时通常返回 Movie 实例；这里也兼容部分适配器返回 dict 的情况。"""
    if is_dataclass(result) and not isinstance(result, type):
        return asdict(result), []
    if isinstance(result, dict):
        return result, []
    return None, [f"返回值既不是 dataclass 实例，也不是 dict，而是 {type(result).__name__}"]


def validate_movie_result(result) -> list[str]:
    data, errors = normalize_movie_result(result)
    if errors:
        return errors

    required_fields = ["title", "year", "director", "cast", "rating"]
    for field_name in required_fields:
        if field_name not in data:
            errors.append(f"缺少必填字段: {field_name}")

    if "title" in data and not isinstance(data["title"], str):
        errors.append("title 应该是 string")
    if "year" in data and (not isinstance(data["year"], int) or isinstance(data["year"], bool)):
        errors.append("year 应该是 integer")
    if "director" in data and not isinstance(data["director"], str):
        errors.append("director 应该是 string")
    if "rating" in data and (not isinstance(data["rating"], (int, float)) or isinstance(data["rating"], bool)):
        errors.append("rating 应该是 number")

    cast = data.get("cast")
    if "cast" in data and not isinstance(cast, list):
        errors.append("cast 应该是 array")
    elif isinstance(cast, list):
        for index, actor in enumerate(cast):
            if not isinstance(actor, dict):
                errors.append(f"cast[{index}] 应该是 object")
                continue
            if not isinstance(actor.get("name"), str):
                errors.append(f"cast[{index}].name 应该是 string")
            if not isinstance(actor.get("role"), str):
                errors.append(f"cast[{index}].role 应该是 string")

    return errors


models = {
    "deepseek-v4-flash": init_chat_model(
        model="deepseek-v4-flash",
        extra_body={"thinking": {"type": "disabled"}},
    ),
    "deepseek-v4-pro": init_chat_model(
        model="deepseek-v4-pro",
        extra_body={"thinking": {"type": "disabled"}},
    ),
    "gpt-5.4-mini": init_chat_model(
        model="gpt-5.4-mini",
        model_provider="openai",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        base_url=os.getenv("OPENROUTER_BASE_URL"),
    ),
}


# 使用较弱提示词，不主动列出字段，用来测试模型/适配器是否真的会按 schema 补齐必填字段。
prompt = "请介绍电影《盗梦空间》"

for model_name, llm in models.items():
    print(f"\n===== {model_name} =====")
    try:
        structured_model = llm.with_structured_output(Movie)
        response = structured_model.invoke(prompt)
        errors = validate_movie_result(response)

        if errors:
            print("[FAIL] dataclass 结构校验失败")
            rich_print(errors)
        else:
            print("[PASS] dataclass 结构校验通过")

        rich_print(response)

    except Exception as e:
        print("[FAIL] 调用失败")
        print(type(e).__name__, e)
